# Forecasting

Compare 3 models on daily order volume:
- Naive (last value)
- Seasonal Naive (repeat last week)
- SARIMA (statistical model with weekly seasonality)

Rule: chronological split, NEVER random, for time series.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import plotly.graph_objects as go
from src.config import DATA_PROCESSED
from src.analytics import daily_orders
from src.forecasting import NaiveForecaster, SeasonalNaiveForecaster, SarimaForecaster, train_test_split, evaluate

orders = pd.read_parquet(DATA_PROCESSED / 'orders.parquet')
daily = daily_orders(orders)
daily['order_date'] = pd.to_datetime(daily['order_date'])
series = daily.set_index('order_date')['order_count'].asfreq('D').fillna(0)
series.plot(title='Daily order volume')

In [ ]:
train, test = train_test_split(series, train_fraction=0.8)
print(f'Train: {len(train)} days, Test: {len(test)} days')

In [ ]:
results = []
for name, model in [('Naive', NaiveForecaster()), ('SeasonalNaive', SeasonalNaiveForecaster(7)), ('SARIMA', SarimaForecaster())]:
    model.fit(train)
    pred = model.predict(len(test))
    results.append({'model': name, **evaluate(test.values, pred.values)})
pd.DataFrame(results).sort_values('wape')

In [ ]:
sarima = SarimaForecaster().fit(train)
pred = sarima.predict(len(test))

fig = go.Figure()
fig.add_scatter(x=train.index, y=train.values, name='Train')
fig.add_scatter(x=test.index, y=test.values, name='Actual (test)')
fig.add_scatter(x=pred.index, y=pred.values, name='SARIMA forecast')
fig.update_layout(title='SARIMA vs Actual')
fig.show()